In [ ]:
import os

try:
    from google.colab import drive
    os.chdir('/content')
    drive.mount('/content/drive', force_remount=True)

    !pip install -q git+https://github.com/compphoto/Intrinsic.git
    !pip install opencv-python
    os.chdir('/content/drive/MyDrive/3dgs_decomposition')
except ModuleNotFoundError:
    pass  # running locally, already in project root

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

In [ ]:
# fetch data and run albedo predictor
from data.get_data import get_data_for_predictors
from predictors.colorful_iid import colorful_iid_predict
from predictors.marigold import marigold_predict
from predictors.diffusion_renderer import diffusion_renderer_predict
from predictors.rgbx.rgbx import rgbx_predict
from predictors.depth_anything import get_depth_map

scenes = [('hypersim', 'ai_001_001'), ('hypersim', 'ai_002_001'), ('mipnerf360', 'bicycle'), ('mipnerf360', 'garden'), ('mipnerf360', 'room')]


In [ ]:
# code to generate 2D priors with colourful iid and save them to drive for future use
for dataset, scene in scenes:
  images = get_data_for_predictors(dataset=dataset, scene=scene)
  albedos, diffuse_shading, residuals = colorful_iid_predict(images)

  os.makedirs(f'colourful_iid/{dataset}/{scene}/albedos', exist_ok=True)
  os.makedirs(f'colourful_iid/{dataset}/{scene}/shading', exist_ok=True)
  os.makedirs(f'colourful_iid/{dataset}/{scene}/residuals', exist_ok=True)

  for idx, image in enumerate(images):
    Image.fromarray((np.clip(albedos[idx], 0, 1) * 255).astype(np.uint8)).save(f'colourful_iid/{dataset}/{scene}/albedos/{os.path.splitext(os.path.basename(image))[0]}.jpg')
    Image.fromarray((np.clip(diffuse_shading[idx], 0, 1) * 255).astype(np.uint8)).save(f'colourful_iid/{dataset}/{scene}/shading/{os.path.splitext(os.path.basename(image))[0]}.jpg')
    Image.fromarray((np.clip(residuals[idx], 0, 1) * 255).astype(np.uint8)).save(f'colourful_iid/{dataset}/{scene}/residuals/{os.path.splitext(os.path.basename(image))[0]}.jpg')
    np.save(f'colourful_iid/{dataset}/{scene}/albedos/{os.path.splitext(os.path.basename(image))[0]}.npy', albedos[idx])
    np.save(f'colourful_iid/{dataset}/{scene}/shading/{os.path.splitext(os.path.basename(image))[0]}.npy', diffuse_shading[idx])
    np.save(f'colourful_iid/{dataset}/{scene}/residuals/{os.path.splitext(os.path.basename(image))[0]}.npy', residuals[idx])

In [ ]:
# code to generate 2D priors with marigold and save them to drive for future use
for dataset, scene in scenes:
  images = get_data_for_predictors(dataset=dataset, scene=scene)
  albedos, diffuse_shading, residuals = marigold_predict(images)

  os.makedirs(f'marigold/{dataset}/{scene}/albedos', exist_ok=True)
  os.makedirs(f'marigold/{dataset}/{scene}/shading', exist_ok=True)
  os.makedirs(f'marigold/{dataset}/{scene}/residuals', exist_ok=True)

  for idx, image in enumerate(images):
    Image.fromarray(albedos[idx]).save(f'marigold/{dataset}/{scene}/albedos/{os.path.splitext(os.path.basename(image))[0]}.jpg')
    Image.fromarray(diffuse_shading[idx]).save(f'marigold/{dataset}/{scene}/shading/{os.path.splitext(os.path.basename(image))[0]}.jpg')
    Image.fromarray(residuals[idx]).save(f'marigold/{dataset}/{scene}/residuals/{os.path.splitext(os.path.basename(image))[0]}.jpg')
    np.save(f'marigold/{dataset}/{scene}/albedos/{os.path.splitext(os.path.basename(image))[0]}.npy', albedos[idx])
    np.save(f'marigold/{dataset}/{scene}/shading/{os.path.splitext(os.path.basename(image))[0]}.npy', diffuse_shading[idx])
    np.save(f'marigold/{dataset}/{scene}/residuals/{os.path.splitext(os.path.basename(image))[0]}.npy', residuals[idx])

In [ ]:
# code to generate 2D priors with diffusion renderer and save them to drive for future use
for dataset, scene in scenes:
  images = get_data_for_predictors(dataset=dataset, scene=scene)
  diffusion_renderer_predict(images, dataset, scene)

In [ ]:
# code to generate 2D priors with rgbx and save them to drive for future use
for dataset, scene in scenes:
  images = get_data_for_predictors(dataset=dataset, scene=scene)
  albedos, diffuse_shading, residuals = rgbx_predict(images, cache_dir='/content/model_cache')


  os.makedirs(f'rgbx/{dataset}/{scene}/albedos', exist_ok=True)
  os.makedirs(f'rgbx/{dataset}/{scene}/shading', exist_ok=True)
  os.makedirs(f'rgbx/{dataset}/{scene}/residuals', exist_ok=True)

  for idx, image in enumerate(images):
    Image.fromarray(albedos[idx]).save(f'rgbx/{dataset}/{scene}/albedos/{os.path.splitext(os.path.basename(image))[0]}.jpg')
    Image.fromarray(diffuse_shading[idx]).save(f'rgbx/{dataset}/{scene}/shading/{os.path.splitext(os.path.basename(image))[0]}.jpg')
    Image.fromarray(residuals[idx]).save(f'rgbx/{dataset}/{scene}/residuals/{os.path.splitext(os.path.basename(image))[0]}.jpg')
    np.save(f'rgbx/{dataset}/{scene}/albedos/{os.path.splitext(os.path.basename(image))[0]}.npy', albedos[idx])
    np.save(f'rgbx/{dataset}/{scene}/shading/{os.path.splitext(os.path.basename(image))[0]}.npy', diffuse_shading[idx])
    np.save(f'rgbx/{dataset}/{scene}/residuals/{os.path.splitext(os.path.basename(image))[0]}.npy', residuals[idx])

In [ ]:
# run depth anything v2
for dataset, scene in scenes:
    images = get_data_for_predictors(dataset=dataset, scene=scene)
    depth_maps = get_depth_map(images)